In [2]:
"""
Decision Tree from scratch (entropy + information gain)
+ the same thing done the easy way with scikit-learn.

Dataset: the "Play Tennis" example we worked out by hand.
"""

import math
from collections import Counter

# ---------------------------------------------------------
# 1. The dataset (same one from the manual example)
# ---------------------------------------------------------
data = [
    {"Outlook": "Sunny",    "Play": "No"},
    {"Outlook": "Sunny",    "Play": "No"},
    {"Outlook": "Overcast", "Play": "Yes"},
    {"Outlook": "Rain",     "Play": "Yes"},
    {"Outlook": "Rain",     "Play": "Yes"},
    {"Outlook": "Rain",     "Play": "No"},
    {"Outlook": "Overcast", "Play": "Yes"},
    {"Outlook": "Sunny",    "Play": "Yes"},
]


# ---------------------------------------------------------
# 2. Entropy function
#    entropy(S) = -sum( p_i * log2(p_i) )  for each class i
# ---------------------------------------------------------
def entropy(rows, label_key="Play"):
    total = len(rows)
    if total == 0:
        return 0

    counts = Counter(row[label_key] for row in rows)  # e.g. {"Yes": 5, "No": 3}

    e = 0
    for count in counts.values():
        p = count / total
        e -= p * math.log2(p)

    return e


# ---------------------------------------------------------
# 3. Information Gain function
#    IG(S, feature) = entropy(S) - weighted_avg_entropy(splits)
# ---------------------------------------------------------
def information_gain(rows, feature_key, label_key="Play"):
    total_entropy = entropy(rows, label_key)

    # group rows by the feature's value (Sunny / Overcast / Rain)
    groups = {}
    for row in rows:
        value = row[feature_key]
        groups.setdefault(value, []).append(row)

    # weighted average entropy after the split
    weighted_entropy = 0
    for group_rows in groups.values():
        weight = len(group_rows) / len(rows)
        weighted_entropy += weight * entropy(group_rows, label_key)

    gain = total_entropy - weighted_entropy
    return gain, groups


# ---------------------------------------------------------
# 4. Run it and print step by step (matches the manual example)
# ---------------------------------------------------------
if __name__ == "__main__":
    print("=== Manual entropy / information gain ===\n")

    base_entropy = entropy(data)
    print(f"Entropy of full dataset: {base_entropy:.3f}")

    gain, groups = information_gain(data, "Outlook")

    for value, rows in groups.items():
        print(f"  Subgroup '{value}': {len(rows)} rows, "
              f"entropy = {entropy(rows):.3f}")

    print(f"\nInformation Gain for 'Outlook': {gain:.3f}")

    # -------------------------------------------------------
    # 5. Now the same idea using scikit-learn (the real-world way)
    # -------------------------------------------------------
    print("\n=== scikit-learn version ===\n")

    from sklearn.tree import DecisionTreeClassifier, export_text
    from sklearn.preprocessing import LabelEncoder
    import pandas as pd

    df = pd.DataFrame(data)

    # sklearn needs numbers, not text, so we encode the categories
    le_outlook = LabelEncoder()
    df["Outlook_enc"] = le_outlook.fit_transform(df["Outlook"])  # Sunny/Overcast/Rain -> 0/1/2

    X = df[["Outlook_enc"]]   # features
    y = df["Play"]            # label

    # criterion="entropy" tells sklearn to use the same math we did by hand
    # (default is "gini", which is a similar but slightly different impurity measure)
    clf = DecisionTreeClassifier(criterion="entropy", random_state=0)
    clf.fit(X, y)

    print(export_text(clf, feature_names=["Outlook_enc"]))
    print("Mapping used:", dict(zip(le_outlook.classes_,
                                     le_outlook.transform(le_outlook.classes_))))

=== Manual entropy / information gain ===

Entropy of full dataset: 0.954
  Subgroup 'Sunny': 3 rows, entropy = 0.918
  Subgroup 'Overcast': 2 rows, entropy = 0.000
  Subgroup 'Rain': 3 rows, entropy = 0.918

Information Gain for 'Outlook': 0.266

=== scikit-learn version ===

|--- Outlook_enc <= 0.50
|   |--- class: Yes
|--- Outlook_enc >  0.50
|   |--- Outlook_enc <= 1.50
|   |   |--- class: Yes
|   |--- Outlook_enc >  1.50
|   |   |--- class: No

Mapping used: {'Overcast': np.int64(0), 'Rain': np.int64(1), 'Sunny': np.int64(2)}
